# 🚑 Limpieza de datos — Llamadas de Urgencias y Emergencias, Línea 123 (Bogotá)

**Autor:** Daniela Vargas  
**Herramientas:** Python (Pandas)  
**Fuente:** [Datos Abiertos Bogotá — Secretaría Distrital de Salud](https://datosabiertos.bogota.gov.co/dataset/llamadas-de-urgencias-y-emergencias-que-ingresan-a-traves-de-la-linea-123)  
**Periodo:** Enero – Junio 2026

## Objetivo

Descargar, combinar y limpiar los registros mensuales de llamadas de urgencias y emergencias que ingresan a la línea 123 de Bogotá, dejando el dataset listo para análisis y visualización en Power BI.


## 1. Descarga de datos

In [1]:
"""
Descarga y combina los datasets mensuales de "Llamadas de Urgencias y
Emergencias - Línea 123" (Secretaría Distrital de Salud, Bogotá).

Fuente: https://datosabiertos.bogota.gov.co/dataset/llamadas-de-urgencias-y-emergencias-que-ingresan-a-traves-de-la-linea-123
"""

import pandas as pd
import requests
from io import StringIO

# Diccionario: nombre del mes -> URL de descarga directa
# (estas URLs se sacan directo de los botones "Descargar" de cada mes en el portal)
urls = {
    "2026-01": "https://datosabiertos.bogota.gov.co/dataset/b99881e0-8750-4fa8-af6c-3f30c9e38708/resource/7fe6ef19-cd32-4050-b354-6871fb38d65a/download/llamadas123_enero2026.csv",
    "2026-02": "https://datosabiertos.bogota.gov.co/dataset/b99881e0-8750-4fa8-af6c-3f30c9e38708/resource/23e406bd-0bc5-401a-8e62-576c0991c6fd/download/llamadas_123_febrero_2026.csv",
    "2026-03": "https://datosabiertos.bogota.gov.co/dataset/b99881e0-8750-4fa8-af6c-3f30c9e38708/resource/0064be65-89eb-4fbc-bcb3-438f8dba0a0c/download/llamasdaslinea123.csv",
    "2026-04": "https://datosabiertos.bogota.gov.co/dataset/b99881e0-8750-4fa8-af6c-3f30c9e38708/resource/af4ea695-bbfc-4d0a-be2c-7c061ef72c92/download/linea123_abril_2026.csv",
    "2026-05": "https://datosabiertos.bogota.gov.co/dataset/b99881e0-8750-4fa8-af6c-3f30c9e38708/resource/1cc2aefb-0fc6-4962-8eed-1b1d76349dc4/download/llamadas123.csv",
    "2026-06": "https://datosabiertos.bogota.gov.co/dataset/b99881e0-8750-4fa8-af6c-3f30c9e38708/resource/04a15ba0-0ef4-4279-bbca-15a253d118c9/download/llamadas123.csv",
}

dataframes = []

for mes, url in urls.items():
    print(f"Descargando {mes}...")
    response = requests.get(url)

    # El portal de Bogotá publica estos CSV separados por ";" y en
    # codificación cp850 (IBM850, típica de exports de sistemas antiguos
    # en Latinoamérica) — NO utf-8 ni latin-1.
    df = pd.read_csv(
        StringIO(response.content.decode("cp850")),
        sep=";",
    )

    df["mes_origen"] = mes  # trazabilidad: de qué archivo vino cada fila
    dataframes.append(df)
    print(f"  -> {len(df)} filas cargadas, {df.shape[1]} columnas")

# Combina todos los meses en un solo DataFrame
df_completo = pd.concat(dataframes, ignore_index=True)

print(f"\nTotal combinado: {len(df_completo)} filas, {df_completo.shape[1]} columnas")
print(df_completo.head())

# Guarda el crudo combinado (antes de limpiar) en tu carpeta raw/
df_completo.to_csv("llamadas123_2026_enero_junio_raw.csv", index=False)

Descargando 2026-01...
  -> 9450 filas cargadas, 11 columnas
Descargando 2026-02...
  -> 8963 filas cargadas, 11 columnas
Descargando 2026-03...
  -> 9862 filas cargadas, 11 columnas
Descargando 2026-04...
  -> 9310 filas cargadas, 11 columnas
Descargando 2026-05...
  -> 9281 filas cargadas, 11 columnas
Descargando 2026-06...
  -> 9206 filas cargadas, 11 columnas

Total combinado: 56072 filas, 11 columnas
  NUMERO_INCIDENTE FECHA_INICIO_DESPLAZAMIENTO_MOVIL CODIGO_LOCALIDAD  \
0  CRU-00000003-26               2026-01-01 19:25:31               14   
1  CRU-00000003-26               2026-01-01 00:03:06               14   
2  CRU-00000009-26               2026-01-01 00:24:29                7   
3  CRU-00000016-26               2026-01-01 00:16:28                6   
4  CRU-00000019-26               2026-01-01 00:44:05               11   

      LOCALIDAD  EDAD UNIDAD     GENERO                TIPO_INCIDENTE  \
0  LOS MÁRTIRES  71.0   Años  MASCULINO       ACOMPAÑAMIENTO A EVENTO   
1  LOS

## 2. Exploración de datos

In [2]:
#Resumen del dataframe incluyendo el número de filas y columnas, la presencia de valores nulos y los tipos de datos de cada columna.
df_completo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56072 entries, 0 to 56071
Data columns (total 11 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   NUMERO_INCIDENTE                   56072 non-null  object 
 1   FECHA_INICIO_DESPLAZAMIENTO_MOVIL  56072 non-null  object 
 2   CODIGO_LOCALIDAD                   56072 non-null  object 
 3   LOCALIDAD                          56072 non-null  object 
 4   EDAD                               37058 non-null  float64
 5   UNIDAD                             37060 non-null  object 
 6   GENERO                             37060 non-null  object 
 7   TIPO_INCIDENTE                     56072 non-null  object 
 8   PRIORIDAD_FINAL                    56072 non-null  object 
 9   RECEPCION                          30948 non-null  object 
 10  mes_origen                         56072 non-null  object 
dtypes: float64(1), object(10)
memory usage: 4.7+ MB


## 3. Confirmar nulos y revisar categorías de texto

In [3]:
# Confirma los nulos por columna
print(df_completo.isnull().sum())
print()

# Revisa si EDAD, UNIDAD y GENERO faltan siempre juntos (misma fila)
print(df_completo[df_completo['EDAD'].isnull()][['UNIDAD', 'GENERO']].isnull().sum())
print()

# Revisa las categorías únicas de las columnas de texto clave
# (para detectar inconsistencias de mayúsculas/minúsculas o espacios)
print("LOCALIDAD:", df_completo['LOCALIDAD'].unique())
print()
print("UNIDAD:", df_completo['UNIDAD'].unique())
print()
print("GENERO:", df_completo['GENERO'].unique())
print()
print("PRIORIDAD_FINAL:", df_completo['PRIORIDAD_FINAL'].unique())

NUMERO_INCIDENTE                         0
FECHA_INICIO_DESPLAZAMIENTO_MOVIL        0
CODIGO_LOCALIDAD                         0
LOCALIDAD                                0
EDAD                                 19014
UNIDAD                               19012
GENERO                               19012
TIPO_INCIDENTE                           0
PRIORIDAD_FINAL                          0
RECEPCION                            25124
mes_origen                               0
dtype: int64

UNIDAD    19012
GENERO    19012
dtype: int64

LOCALIDAD: ['LOS MÁRTIRES' 'BOSA' 'TUNJUELITO' 'SUBA' 'SANTA FE' 'KENNEDY'
 'PUENTE ARANDA' 'CIUDAD BOLÍVAR' 'CHAPINERO' 'ENGATIVÁ' 'USAQUÉN'
 'SAN CRISTÓBAL' 'TEUSAQUILLO' 'BARRIOS UNIDOS' 'ANTONIO NARIÑO'
 'RAFAEL URIBE URIBE' 'USME' 'FONTIBÓN' 'LA CANDELARIA' 'SIN_D' 'SUMAPAZ'
 'ENGATIVA' 'ANTONIO NARIÐO' 'FONTIBON' 'SAN CRISTOBAL' 'LOS MARTIRES'
 'FUERA_DE_BOGOTA' 'BARRIOS UNIDOS\xa0' 'CIUDAD BOLIVAR'
 'ANTONIO NARIÐO\xa0']

UNIDAD: ['Años' 'Meses' nan 'Dias'

## 4. Análisis de las columnas

In [4]:
# Cuántas filas tiene cada valor de LOCALIDAD (para ver el peso real del problema)
print(df_completo['LOCALIDAD'].value_counts())
print()

# Lo mismo para UNIDAD
print(df_completo['UNIDAD'].value_counts())
print()

# Lo mismo para PRIORIDAD_FINAL (para ver si los números o el texto predominan)
print(df_completo['PRIORIDAD_FINAL'].value_counts())
print()

# Bono: revisa si los 2 casos raros de EDAD null (pero UNIDAD/GENERO no null) existen
print(df_completo[df_completo['EDAD'].isnull() & df_completo['UNIDAD'].notnull()])

LOCALIDAD
KENNEDY               8438
SUBA                  5675
BOSA                  4479
PUENTE ARANDA         3410
ENGATIVA              2836
ENGATIVÁ              2798
RAFAEL URIBE URIBE    2767
CIUDAD BOLÍVAR        2715
USAQUÉN               2348
USME                  2210
SANTA FE              1712
TEUSAQUILLO           1678
TUNJUELITO            1594
SAN CRISTOBAL         1445
SAN CRISTÓBAL         1440
CHAPINERO             1402
FONTIBON              1392
CIUDAD BOLIVAR        1311
FONTIBÓN              1281
BARRIOS UNIDOS        1249
LOS MARTIRES          1043
LOS MÁRTIRES          1014
ANTONIO NARIÑO         500
LA CANDELARIA          450
ANTONIO NARIÐO         383
BARRIOS UNIDOS         270
ANTONIO NARIÐO         168
SIN_D                   52
FUERA_DE_BOGOTA          9
SUMAPAZ                  3
Name: count, dtype: int64

UNIDAD
Años     30228
A±os      6646
Meses      150
Dias        20
Horas       16
Name: count, dtype: int64

PRIORIDAD_FINAL
Alta       25469
Critica    

## 5. Búsqueda de códigos numéricos de prioridad

In [5]:
# ¿Los códigos numéricos están aislados a un solo mes?
print(df_completo.groupby('mes_origen')['PRIORIDAD_FINAL'].apply(lambda x: x.unique()))
url_metadata = "https://datosabiertos.bogota.gov.co/dataset/b99881e0-8750-4fa8-af6c-3f30c9e38708/resource/3d9813d6-897e-4ea8-a891-7514106bf180/download/metadato_llamadas123.csv"
response = requests.get(url_metadata)
df_metadata = pd.read_csv(StringIO(response.content.decode("cp850")), sep=";")
print(df_metadata.to_string())


mes_origen
2026-01    [Baja, Critica, Alta, Media]
2026-02    [Baja, Critica, Alta, Media]
2026-03                    [4, 2, 1, 3]
2026-04    [Baja, Alta, Critica, Media]
2026-05    [Baja, Alta, Critica, Media]
2026-06    [Baja, Alta, Critica, Media]
Name: PRIORIDAD_FINAL, dtype: object
                              NOMBRE                                                                                                                             DESCRIPCION
0                   NUMERO_INCIDENTE         Es la secuencia numerica que se asigna a los incidentes para cada una de las agencias en la plataforma tecnologica Premier One.
1  FECHA_INICIO_DESPLAZAMIENTO_MOVIL                                             Es la fecha el cual se inicia el desplazamiento de la ambulancia al sitio de la emergencia.
2                   CODIGO LOCALIDAD  Es el codigo de las 20 localidades de la ciudad de bogota segun el codigo perteneciente a cada una de ellas designado por el distrito 
3                   

## 6. Tratamiento de PRIORIDAD_FINAL no verificable (marzo 2026)

In [6]:
# --------------------------------------------------------------------
# NOTA DE CALIDAD DE DATOS: PRIORIDAD_FINAL en marzo 2026
# --------------------------------------------------------------------
# El archivo de marzo 2026 vino con PRIORIDAD_FINAL codificada como
# números (1, 2, 3, 4) en vez de texto (Critica, Alta, Media, Baja).
# El metadato oficial del portal no incluye el diccionario de
# equivalencia numérica, así que NO se infiere el mapeo (aunque las
# proporciones sugieren un patrón probable, inferirlo sin confirmación
# oficial arriesga análisis incorrectos). Se marcan esas filas como
# "No verificable" para excluirlas de análisis por prioridad, pero se
# conservan en el dataset para conteos generales.

es_codigo_numerico = df_completo['PRIORIDAD_FINAL'].isin([1, 2, 3, 4, "1", "2", "3", "4"])
print(f"Filas con PRIORIDAD_FINAL no verificable (marzo 2026): {es_codigo_numerico.sum()}")

df_completo['PRIORIDAD_FINAL'] = df_completo['PRIORIDAD_FINAL'].where(
    ~es_codigo_numerico, "No verificable"
)

Filas con PRIORIDAD_FINAL no verificable (marzo 2026): 9862


## 7. Unificación de LOCALIDAD y UNIDAD

In [7]:
# UNIFICACIÓN DE LOCALIDAD
# La misma localidad aparece escrita de varias formas: con tilde, sin
# tilde, y con errores de encoding (Ð en vez de Ñ). Se unifica todo a un solo nombre estándar (con tilde correcta) por localidad.

mapeo_localidad = {
    'ENGATIVA': 'ENGATIVÁ',
    'FONTIBON': 'FONTIBÓN',
    'SAN CRISTOBAL': 'SAN CRISTÓBAL',
    'CIUDAD BOLIVAR': 'CIUDAD BOLÍVAR',
    'LOS MARTIRES': 'LOS MÁRTIRES',
    'ANTONIO NARIÐO': 'ANTONIO NARIÑO',
}

# Primero quita espacios invisibles (\xa0) y espacios normales de sobra
df_completo['LOCALIDAD'] = df_completo['LOCALIDAD'].str.strip().str.replace('\xa0', '', regex=False)

# Luego aplica el mapeo de unificación
df_completo['LOCALIDAD'] = df_completo['LOCALIDAD'].replace(mapeo_localidad)

print("LOCALIDAD después de unificar:")
print(df_completo['LOCALIDAD'].value_counts())
print()

# UNIFICACIÓN DE UNIDAD
# 'Años' y 'A±os' son el mismo valor, solo con error de encoding.

df_completo['UNIDAD'] = df_completo['UNIDAD'].replace({'A±os': 'Años'})

print("UNIDAD después de unificar:")
print(df_completo['UNIDAD'].value_counts())

LOCALIDAD después de unificar:
LOCALIDAD
KENNEDY               8438
SUBA                  5675
ENGATIVÁ              5634
BOSA                  4479
CIUDAD BOLÍVAR        4026
PUENTE ARANDA         3410
SAN CRISTÓBAL         2885
RAFAEL URIBE URIBE    2767
FONTIBÓN              2673
USAQUÉN               2348
USME                  2210
LOS MÁRTIRES          2057
SANTA FE              1712
TEUSAQUILLO           1678
TUNJUELITO            1594
BARRIOS UNIDOS        1519
CHAPINERO             1402
ANTONIO NARIÑO        1051
LA CANDELARIA          450
SIN_D                   52
FUERA_DE_BOGOTA          9
SUMAPAZ                  3
Name: count, dtype: int64

UNIDAD después de unificar:
UNIDAD
Años     36874
Meses      150
Dias        20
Horas       16
Name: count, dtype: int64


## 8. Conversión de columnas de fecha

In [8]:
# CONVERSIÓN DE FECHAS
# HALLAZGO DE CALIDAD DE DATOS: estas columnas mezclan DOS formatos de
# fecha distintos según el mes de origen: formato ISO (2026-01-01
# 19:25:31) en algunos meses, y formato colombiano día/mes/año
# (3/04/2026 8:46) en otros. Se usa format='mixed', que le permite a
# Pandas detectar el formato de cada fila individualmente, con
# dayfirst=True para resolver bien los casos ambiguos del formato
# día/mes/año.

df_completo['FECHA_INICIO_DESPLAZAMIENTO_MOVIL'] = pd.to_datetime(
    df_completo['FECHA_INICIO_DESPLAZAMIENTO_MOVIL'],
    format='mixed',
    dayfirst=True,
    errors='coerce'
)

df_completo['RECEPCION'] = pd.to_datetime(
    df_completo['RECEPCION'],
    format='mixed',
    dayfirst=True,
    errors='coerce'
)

print("Tipos de dato después de la conversión:")
print(df_completo[['FECHA_INICIO_DESPLAZAMIENTO_MOVIL', 'RECEPCION']].dtypes)
print()

print("Nulos en FECHA_INICIO_DESPLAZAMIENTO_MOVIL:", df_completo['FECHA_INICIO_DESPLAZAMIENTO_MOVIL'].isnull().sum())
print("Nulos en RECEPCION:", df_completo['RECEPCION'].isnull().sum())

Tipos de dato después de la conversión:
FECHA_INICIO_DESPLAZAMIENTO_MOVIL    datetime64[ns]
RECEPCION                            datetime64[ns]
dtype: object

Nulos en FECHA_INICIO_DESPLAZAMIENTO_MOVIL: 0
Nulos en RECEPCION: 25124


## 9. Investigación de nulos en GENERO

In [9]:
# ¿Qué tipo de incidentes son los que no tienen GENERO registrado?
print("Top 15 tipos de incidente SIN género registrado:")
print(df_completo[df_completo['GENERO'].isnull()]['TIPO_INCIDENTE'].value_counts().head(15))
print()

# Compara contra el top de incidentes CON género registrado, para ver si es el mismo patrón o distinto
print("Top 15 tipos de incidente CON género registrado:")
print(df_completo[df_completo['GENERO'].notnull()]['TIPO_INCIDENTE'].value_counts().head(15))

Top 15 tipos de incidente SIN género registrado:
TIPO_INCIDENTE
HERIDO - HERIDOS ACCIDENTALES                           5268
INCONSCIEN - INCONSCIENTE O PARO CARDIORRESPIRATORIO    2262
INTSUI - INTENTO DE SUICIDIO                            1618
EVERES - EVENTO RESPIRATORIO                            1420
TRASTMENT - TRASTORNO MENTAL                            1357
ENFERMO                                                 1272
CONVULSION - CONVULSION                                  988
CONVULSIÓN - CONVULSIÓN                                  974
DOLTOR - DOLOR TORÁCICO                                  534
ACV - ACCIDENTE CEREBRO VASCULAR                         409
CAIALT - CAÍDA DE ALTURA                                 314
DOLTOR - DOLOR TORACICO                                  277
ACOMPAÐAMIENTO A EVENTO                                  262
CAIALT - CAIDA DE ALTURA                                 260
AMESUI  - AMENAZA DE SUICIDIO                            242
Name: count, dtype: i

In [10]:
# ¿Los nulos de GENERO se concentran en algún mes en particular?
print("Nulos de GENERO por mes:")
print(df_completo[df_completo['GENERO'].isnull()]['mes_origen'].value_counts())
print()
print("Total de filas por mes (para comparar proporciones):")
print(df_completo['mes_origen'].value_counts())
print()

# ¿Los casos sin GENERO también son casos sin RECEPCION? (¿es el mismo grupo de "casos no completados"?)
print("De los casos SIN género, cuántos tampoco tienen RECEPCION:")
print(df_completo[df_completo['GENERO'].isnull()]['RECEPCION'].isnull().sum(), "de", df_completo['GENERO'].isnull().sum())

Nulos de GENERO por mes:
mes_origen
2026-06    3262
2026-05    3259
2026-03    3194
2026-01    3171
2026-04    3127
2026-02    2999
Name: count, dtype: int64

Total de filas por mes (para comparar proporciones):
mes_origen
2026-03    9862
2026-01    9450
2026-04    9310
2026-05    9281
2026-06    9206
2026-02    8963
Name: count, dtype: int64

De los casos SIN género, cuántos tampoco tienen RECEPCION:
19012 de 19012


## 10. Pasos finales: nulos, duplicados y exportación

In [11]:
# Los nulos de GENERO, EDAD, UNIDAD y RECEPCION están correlacionados:
# son casos donde la llamada no se completó (no se llegó a registrar
# datos del paciente). Se conservan las filas mismo, pero se marcan
# explícitamente en vez de dejar el nulo ambiguo.

df_completo['GENERO'] = df_completo['GENERO'].fillna('No registrado')
df_completo['UNIDAD'] = df_completo['UNIDAD'].fillna('No registrado')
# EDAD y RECEPCION se dejan como nulos (NaN/NaT) porque son numéricas/fecha,
# no tiene sentido rellenarlas con texto.

# Duplicados
print("Filas duplicadas:", df_completo.duplicated().sum())
df_completo = df_completo.drop_duplicates()

# Unifica TIPO_INCIDENTE (mismo problema de tildes que LOCALIDAD)
mapeo_incidente = {
    'CONVULSION - CONVULSION': 'CONVULSIÓN - CONVULSIÓN',
    'DOLTOR - DOLOR TORACICO': 'DOLTOR - DOLOR TORÁCICO',
    'CAIALT - CAIDA DE ALTURA': 'CAIALT - CAÍDA DE ALTURA',
    'ACOMPAÐAMIENTO A EVENTO': 'ACOMPAÑAMIENTO A EVENTO',
    'SINTOGASTR - SINTOMAS GASTROINTESTINALES': 'SINTOGASTR - SÍNTOMAS GASTROINTESTINALES',
}
df_completo['TIPO_INCIDENTE'] = df_completo['TIPO_INCIDENTE'].str.strip().replace(mapeo_incidente)

# Exporta el dataset limpio final
df_completo.to_csv('llamadas123_2026_enero_junio_clean.csv', index=False)
print(f"\nDataset final: {len(df_completo)} filas, {df_completo.shape[1]} columnas")
print(df_completo.info())

Filas duplicadas: 277

Dataset final: 55795 filas, 11 columnas
<class 'pandas.core.frame.DataFrame'>
Index: 55795 entries, 0 to 56071
Data columns (total 11 columns):
 #   Column                             Non-Null Count  Dtype         
---  ------                             --------------  -----         
 0   NUMERO_INCIDENTE                   55795 non-null  object        
 1   FECHA_INICIO_DESPLAZAMIENTO_MOVIL  55795 non-null  datetime64[ns]
 2   CODIGO_LOCALIDAD                   55795 non-null  object        
 3   LOCALIDAD                          55795 non-null  object        
 4   EDAD                               36940 non-null  float64       
 5   UNIDAD                             55795 non-null  object        
 6   GENERO                             55795 non-null  object        
 7   TIPO_INCIDENTE                     55795 non-null  object        
 8   PRIORIDAD_FINAL                    55795 non-null  object        
 9   RECEPCION                          30915 non-nu

**Descarga de archivos**

## 11. Resumen de resultados

**Dataset final:** 55,795 filas × 11 columnas (enero–junio 2026)

**Problemas de calidad detectados y resueltos:**
- Separador `;` y encoding `cp850` corregidos en la descarga
- 277 filas duplicadas eliminadas
- `LOCALIDAD` y `TIPO_INCIDENTE` unificados (variantes de tildes/encoding)
- `UNIDAD` unificado (`Años` / `A±os`)
- `GENERO` y `UNIDAD` nulos marcados como `"No registrado"` (19,012 casos de llamadas no completadas, confirmado porque también carecían de `RECEPCION`)
- `PRIORIDAD_FINAL` de marzo 2026 marcado como `"No verificable"` (códigos numéricos sin diccionario oficial de equivalencia)
- Fechas convertidas correctamente (el dataset mezclaba formato ISO y formato colombiano)

**Archivo de salida:** `llamadas123_2026_enero_junio_clean.csv`
